# Lab: Perturbing a hybrid AI weather model (NeuralGCM)

**NeuralGCM** (Google, Kochkov et al. 2024, *Nature*) is a *hybrid* model: a conventional dynamical core solves the primitive equations on the sphere, and neural networks replace the physics parameterizations (clouds, convection, radiation, boundary layer, ...). The networks were trained on ERA5.

Because the large-scale dynamics are computed explicitly, a perturbation you add has to evolve through real advection, adjustment and wave propagation. In this lab you will:

1. Run a 10-day forecast and compare it with ERA5.
2. Test the **butterfly effect**: add a tiny temperature perturbation and watch the error grow.
3. Add a **localized warm anomaly** and follow the dynamical response.
4. Change the **sea-surface temperature** (an El Niño–like warm patch) and look for a remote response.
5. Design **your own experiment**.

### Before you start
* `Runtime → Change runtime type → T4 GPU` (or any GPU), then `Runtime → Run all` up to Part 1.
* The first forecast takes ~1–2 minutes extra while JAX compiles the model. Later runs are much faster.
* Cells marked **✏️** are the ones you are meant to edit.

In [ ]:
#@title Install packages (~1 min)
!pip install -q neuralgcm gcsfs cartopy
# If the import cell below fails with a version error, uncomment this line, run it, then
# restart the session (Runtime -> Restart session) and continue from the import cell:
# !pip install -q "neuralgcm==1.2.2" "dinosaur==1.3.6"

In [ ]:
#@title Imports and GPU check
import pickle
import time

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gcsfs
import jax
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils

import neuralgcm

print("JAX devices:", jax.devices())
if jax.default_backend() != "gpu":
    print("\n*** No GPU found. Use Runtime -> Change runtime type -> GPU, then rerun. ***")

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "figure.dpi": 110})
G = 9.80665  # m s^-2, converts geopotential to geopotential height

## Setup: load the model and the initial state

Available checkpoints (all trained on ERA5):

| checkpoint | grid spacing | notes |
|---|---|---|
| `v1/deterministic_2_8_deg.pkl` | 2.8° | fastest, recommended for this lab |
| `v1/deterministic_1_4_deg.pkl` | 1.4° | sharper, ~4× slower |
| `v1/stochastic_1_4_deg.pkl` | 1.4° | stochastic physics: each random seed gives a different ensemble member |

The 0.7° model also exists but likely needs more GPU memory than a free-tier T4.

In [ ]:
#@title ✏️ Choose the model
MODEL_NAME = "v1/deterministic_2_8_deg.pkl"  #@param ["v1/deterministic_2_8_deg.pkl", "v1/deterministic_1_4_deg.pkl", "v1/stochastic_1_4_deg.pkl"]

gcs = gcsfs.GCSFileSystem(token="anon")
with gcs.open(f"gs://neuralgcm/models/{MODEL_NAME}", "rb") as f:
    ckpt = pickle.load(f)
model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)

print("Prognostic inputs:", model.input_variables)
print("Forcings (held fixed during the forecast):", model.forcing_variables)

The initial state comes from **ERA5** (Google's public ARCO-ERA5 copy on Cloud Storage, no account needed). It is on a 0.25° grid with 37 pressure levels, so we average it conservatively onto the model grid.

The default start date, **8 Feb 2021**, is just before the February 2021 cold-air outbreak over Texas, a large-amplitude midlatitude wave pattern. Any date from 1940 to 2025 works (00 UTC is safest).

In [ ]:
#@title ✏️ Start date, then load ERA5 (~1–2 min)
START_TIME = "2021-02-08T00:00"  #@param {type:"string"}
T0 = np.datetime64(START_TIME)

ERA5_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
era5 = xr.open_zarr(ERA5_PATH, chunks=None, storage_options=dict(token="anon"))

tic = time.time()
raw = (
    era5[model.input_variables + model.forcing_variables]
    # The model expects forcings (SST, sea ice) lagged by 24 h, as in training
    .pipe(xarray_utils.selective_temporal_shift,
          variables=model.forcing_variables, time_shift="24 hours")
    .sel(time=[T0])
    .compute()
)

era5_grid = spherical_harmonic.Grid(
    latitude_nodes=era5.sizes["latitude"],
    longitude_nodes=era5.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(era5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(era5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True
)


def to_model_grid(ds):
    return xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(ds, regridder))


regridded = xarray_utils.regrid(raw, regridder)
OCEAN = regridded["sea_surface_temperature"].isel(time=0).notnull()  # SST is NaN over land
IC = xarray_utils.fill_nan_with_nearest(regridded)  # initial condition on the model grid
del raw, regridded
print(f"Loaded and regridded in {time.time() - tic:.0f} s")
IC

In [ ]:
#@title Helper functions: running the model, diagnostics, maps
def run_forecast(ic, days, output_hours=24, seed=0):
    # Run NeuralGCM from the dataset `ic` (one time) for `days` days.
    # Forcings (SST, sea ice) are held at their initial values (persistence).
    # Returns an xarray Dataset with dimension `lead_day` (0 = initial state).
    n_out = days * 24 // output_hours + 1
    inputs = model.inputs_from_xarray(ic.isel(time=0))
    forcings = model.forcings_from_xarray(ic.isel(time=0))
    state = model.encode(inputs, forcings, jax.random.key(seed))
    _, preds = model.unroll(
        state,
        model.forcings_from_xarray(ic.head(time=1)),
        steps=n_out,
        timedelta=np.timedelta64(output_hours, "h"),
        start_with_input=True,
    )
    lead_hours = np.arange(n_out) * output_hours
    out = model.data_to_xarray(preds, times=lead_hours)
    return out.rename(time="lead_day").assign_coords(lead_day=lead_hours / 24)


def era5_truth(days, variables=("geopotential", "temperature"), levels=(500, 850)):
    # ERA5 at the forecast's daily valid times, regridded to the model grid.
    times = T0 + np.arange(days + 1) * np.timedelta64(1, "D")
    ds = era5[list(variables)].sel(time=times, level=list(levels)).compute()
    ds = to_model_grid(ds)
    return ds.rename(time="lead_day").assign_coords(lead_day=np.arange(days + 1))


def z500(ds):
    # 500 hPa geopotential height (m)
    return ds["geopotential"].sel(level=500) / G


def area_rms(da):
    # cos(latitude)-weighted RMS over the globe
    w = np.cos(np.deg2rad(da.latitude))
    return np.sqrt((da ** 2).weighted(w).mean(("latitude", "longitude")))


def gaussian_blob(lat0, lon0, radius_km, grid=None):
    # exp(-(d/R)^2), d = great-circle distance from (lat0, lon0)
    grid = IC if grid is None else grid
    lat, lon = np.deg2rad(grid.latitude), np.deg2rad(grid.longitude)
    la0, lo0 = np.deg2rad(lat0), np.deg2rad(lon0)
    cosd = np.sin(lat) * np.sin(la0) + np.cos(lat) * np.cos(la0) * np.cos(lon - lo0)
    dist_km = 6371.0 * np.arccos(np.clip(cosd, -1.0, 1.0))
    return np.exp(-((dist_km / radius_km) ** 2))


def add_to(ds, var, anomaly):
    # Return a copy of ds with `anomaly` (broadcastable DataArray) added to `var`
    out = ds.copy(deep=True)
    out[var] = (ds[var] + anomaly).transpose(*ds[var].dims)
    return out


def _cyclic(da):
    # Repeat the first longitude at +360 so maps have no seam
    first = da.isel(longitude=[0])
    return xr.concat([da, first.assign_coords(longitude=first.longitude + 360)], "longitude")


def plot_maps(fields, titles, cmap="RdBu_r", vmax=None, contours=None, cbar_label="",
              ncols=2, central_longitude=180):
    # Panel of global maps. `contours` (optional) is a matching list of fields drawn as black lines.
    n = len(fields)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 3.4 * nrows), squeeze=False,
                             subplot_kw={"projection": ccrs.Robinson(central_longitude)})
    if vmax is None:
        vmax = float(max(np.abs(f).max() for f in fields))
    diverging = cmap in ("RdBu_r", "BrBG", "PuOr_r", "coolwarm")
    vmin = -vmax if diverging else float(min(f.min() for f in fields))
    for i, ax in enumerate(axes.flat):
        if i >= n:
            ax.remove()
            continue
        f = _cyclic(fields[i]).transpose("latitude", "longitude")  # pcolormesh/contour need (lat, lon)
        mesh = ax.pcolormesh(f.longitude, f.latitude, f, transform=ccrs.PlateCarree(),
                             cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        if contours is not None:
            c = _cyclic(contours[i]).transpose("latitude", "longitude")  # pcolormesh/contour need (lat, lon)
            ax.contour(c.longitude, c.latitude, c, levels=12, colors="k", linewidths=0.6,
                       transform=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.3")
        ax.set_global()
        ax.set_title(titles[i])
    fig.colorbar(mesh, ax=axes, orientation="horizontal", shrink=0.5, pad=0.04, label=cbar_label)
    plt.show()

---
## Part 1: Control forecast vs. ERA5

Run a 10-day forecast from the ERA5 state and compare 500 hPa heights with what actually happened.

In [ ]:
#@title ✏️ Run the control forecast
FORECAST_DAYS = 10  #@param {type:"integer"}

tic = time.time()
control = run_forecast(IC, FORECAST_DAYS)
truth = era5_truth(FORECAST_DAYS)
print(f"Forecast done in {time.time() - tic:.0f} s (includes JIT compilation on the first run)")

days_to_show = [d for d in (1, 3, 5, FORECAST_DAYS) if d <= FORECAST_DAYS]
fields, titles = [], []
for d in days_to_show:
    fields += [z500(truth).sel(lead_day=d), z500(control).sel(lead_day=d)]
    titles += [f"ERA5, day {d}", f"NeuralGCM, day {d}"]
plot_maps(fields, titles, cmap="viridis", vmax=float(z500(truth).max()),
          cbar_label="500 hPa geopotential height (m)")

In [ ]:
#@title Forecast error vs. lead time
err = area_rms(z500(control) - z500(truth))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(err.lead_day, err, "o-", lw=2)
ax.set(xlabel="Lead time (days)", ylabel="Global RMS error (m)",
       title="Z500 error of the NeuralGCM forecast vs. ERA5")
ax.grid(alpha=0.3)
plt.show()

**Questions**
1. At what lead time do the forecast and ERA5 maps start to look clearly different? Which features are still right at day 10?
2. Why does the error grow with lead time even though the model was trained on this same ERA5 data?

---
## Part 2: The butterfly effect

Add random noise with a standard deviation of only `NOISE_K` kelvin to the temperature at every grid point and level, run again, and measure how far the perturbed forecast drifts from the control.

In a chaotic atmosphere, tiny differences should grow roughly exponentially and then saturate when the two forecasts are as different as two random weather states.

In [ ]:
#@title ✏️ Perturbed forecast
NOISE_K = 0.01  #@param {type:"number"}
BUTTERFLY_DAYS = 15  #@param {type:"integer"}

rng = np.random.default_rng(1)
noise = xr.DataArray(rng.normal(0.0, NOISE_K, IC["temperature"].shape), dims=IC["temperature"].dims)
IC_noisy = add_to(IC, "temperature", noise)

control_long = run_forecast(IC, BUTTERFLY_DAYS)
noisy = run_forecast(IC_noisy, BUTTERFLY_DAYS)
diff = z500(noisy) - z500(control_long)

# A rough "saturation" level: RMS difference between two unrelated states (day 0 vs. last day)
saturation = float(area_rms(z500(control_long).isel(lead_day=-1) - z500(control_long).isel(lead_day=0)))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(diff.lead_day, area_rms(diff), "o-", lw=2, label=f"perturbed - control ({NOISE_K} K noise)")
ax.axhline(saturation, color="0.5", ls="--", label="~ difference between unrelated states")
ax.set(xlabel="Lead time (days)", ylabel="Global RMS Z500 difference (m)",
       title="Growth of a tiny initial perturbation")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
plt.show()

show = [d for d in (1, 3, 6, 10) if d <= BUTTERFLY_DAYS]
plot_maps([diff.sel(lead_day=d) for d in show], [f"Z500 difference, day {d}" for d in show],
          contours=[z500(control_long).sel(lead_day=d) for d in show],
          cbar_label="Perturbed - control Z500 (m); contours: control Z500")

**Questions**
1. Estimate the doubling time of the difference during the period of fastest growth. How does it compare with the ~1–2 day doubling time usually quoted for synoptic-scale errors?
2. Where on the globe do the differences first become large? Why there?
3. Try `NOISE_K = 0.001` and `0.1`. Does the time to saturation change much? What does that imply about improving forecasts by improving observations?
4. Selz & Craig (2023, *GRL*) found that the purely data-driven Pangu-Weather model does *not* amplify very small perturbations the way physics-based models do. Based on your results, does NeuralGCM behave more like a physics model? Why might a dynamical core matter?

---
## Part 3: A localized warm anomaly

Add a Gaussian warm anomaly to the temperature in a layer of the troposphere and follow its evolution. The anomaly is added only to temperature, so the initial state is no longer in hydrostatic or geostrophic balance, and the model has to adjust.

Ideas: a midlatitude anomaly under the jet (default), a tropical anomaly on the equator (compare with the Gill response), or one over the Arctic.

In [ ]:
#@title ✏️ Warm anomaly experiment
BLOB_LAT = 40.0  #@param {type:"number"}
BLOB_LON = 180.0  #@param {type:"number"}
BLOB_RADIUS_KM = 1000.0  #@param {type:"number"}
BLOB_AMPLITUDE_K = 3.0  #@param {type:"number"}
BLOB_BOTTOM_HPA = 1000  #@param {type:"integer"}
BLOB_TOP_HPA = 500  #@param {type:"integer"}
BLOB_DAYS = 10  #@param {type:"integer"}

in_layer = ((IC.level >= BLOB_TOP_HPA) & (IC.level <= BLOB_BOTTOM_HPA)).astype(float)
warm = BLOB_AMPLITUDE_K * gaussian_blob(BLOB_LAT, BLOB_LON, BLOB_RADIUS_KM) * in_layer
IC_warm = add_to(IC, "temperature", warm)

ctrl = control_long if BLOB_DAYS <= BUTTERFLY_DAYS else run_forecast(IC, BLOB_DAYS)
ctrl = ctrl.sel(lead_day=slice(0, BLOB_DAYS))
warm_run = run_forecast(IC_warm, BLOB_DAYS)

show = [d for d in (0, 1, 2, 4, 6, 10) if d <= BLOB_DAYS]
plot_maps([(warm_run - ctrl)["temperature"].sel(level=850, lead_day=d) for d in show],
          [f"T850 difference, day {d}" for d in show],
          cbar_label="Warm - control T850 (K)")
plot_maps([z500(warm_run).sel(lead_day=d) - z500(ctrl).sel(lead_day=d) for d in show],
          [f"Z500 difference, day {d}" for d in show],
          contours=[z500(ctrl).sel(lead_day=d) for d in show],
          cbar_label="Warm - control Z500 (m); contours: control Z500")

**Questions**
1. What happens to the anomaly in the first day? Does the Z500 response look like what you'd expect from the hypsometric equation?
2. In which direction does the signal spread with time? Is it faster than the mean wind? What kind of waves could do that?
3. Move the anomaly to the equator (`BLOB_LAT = 0`). How does the response differ from the midlatitude case?
4. After ~1 week, how much of the difference is the "real" response, and how much is the butterfly effect from Part 2?

---
## Part 4: Changing the sea-surface temperature

NeuralGCM treats SST and sea ice as **forcings**: they are prescribed and held fixed during the forecast. Here we add an El Niño–like warm patch in the central/eastern equatorial Pacific and look for a response far away, such as a PNA-like wave train over North America in northern winter.

The response to SST is small compared with day-to-day weather, and Part 2 showed that forecasts lose their memory of the initial state after ~2 weeks. So we run a small **ensemble** for each case (members differ only by tiny initial noise) and compare **ensemble means**, averaged over the second half of the run.

⏱ This is the slowest part: it runs 2 × `ENSEMBLE_SIZE` forecasts of `SST_DAYS` days.

In [ ]:
#@title ✏️ SST experiment
SST_ANOMALY_K = 2.0  #@param {type:"number"}
SST_CENTER_LAT = 0.0  #@param {type:"number"}
SST_CENTER_LON = 220.0  #@param {type:"number"}
SST_HALF_WIDTH_LAT = 10.0  #@param {type:"number"}
SST_HALF_WIDTH_LON = 40.0  #@param {type:"number"}
SST_DAYS = 30  #@param {type:"integer"}
ENSEMBLE_SIZE = 4  #@param {type:"integer"}

dlon = ((IC.longitude - SST_CENTER_LON + 180) % 360) - 180
patch = np.exp(-((IC.latitude - SST_CENTER_LAT) / SST_HALF_WIDTH_LAT) ** 2
               - (dlon / SST_HALF_WIDTH_LON) ** 2)
sst_anom = (SST_ANOMALY_K * patch).where(OCEAN, 0.0)  # ocean only
IC_sst = add_to(IC, "sea_surface_temperature", sst_anom)

plot_maps([sst_anom], ["Imposed SST anomaly"], cbar_label="K", ncols=1)


def ensemble_mean(ic, days, n):
    members = []
    for m in range(n):
        rng = np.random.default_rng(100 + m)
        eps = xr.DataArray(rng.normal(0.0, 0.01, ic["temperature"].shape), dims=ic["temperature"].dims)
        members.append(run_forecast(add_to(ic, "temperature", eps), days, seed=m))
        print(f"  member {m + 1}/{n} done")
    return xr.concat(members, "member")


tic = time.time()
print("Control ensemble:")
ens_ctrl = ensemble_mean(IC, SST_DAYS, ENSEMBLE_SIZE)
print("Warm-SST ensemble:")
ens_sst = ensemble_mean(IC_sst, SST_DAYS, ENSEMBLE_SIZE)
print(f"Done in {time.time() - tic:.0f} s")

In [ ]:
#@title Ensemble-mean response, averaged over the second half of the run
late = slice(SST_DAYS / 2, SST_DAYS)
resp = (ens_sst - ens_ctrl).sel(lead_day=late).mean(("member", "lead_day"))
# Member-to-member spread of the difference, a rough measure of noise
spread = (ens_sst - ens_ctrl).sel(lead_day=late).mean("lead_day").std("member")

for f, t, lab in [
    (resp["temperature"].sel(level=850), "T850 response", "K"),
    (resp["u_component_of_wind"].sel(level=200), "U200 response", "m/s"),
    (resp["geopotential"].sel(level=500) / G, "Z500 response", "m"),
    (spread["geopotential"].sel(level=500) / G, "Z500 member-to-member spread of the response", "m"),
]:
    plot_maps([f], [f"{t}, days {SST_DAYS // 2}-{SST_DAYS}"], cbar_label=lab, ncols=1,
              cmap="RdBu_r" if "spread" not in t else "viridis")

**Questions**
1. Where is the response largest near the SST anomaly? In the upper troposphere, does it look like a Gill-type pattern?
2. Is there a remote response in the extratropics? Compare its size with the member-to-member spread. Is it robust with only a few members?
3. How would you make this experiment more convincing? (Think about ensemble size, run length, season.)
4. Try a uniform warming instead: set `SST_HALF_WIDTH_LAT` and `SST_HALF_WIDTH_LON` very large (e.g. 1000). Remember the model was trained only on the present climate. Should we trust its response to a climate it has never seen?

---
## Part 5: Your own experiment

Design a perturbation, predict what will happen **before** you run it, then test your prediction. Some ideas:

* Weaken or strengthen the jet: scale `u_component_of_wind` in a latitude band.
* Dry out the atmosphere over a region: scale `specific_humidity`.
* Remove the Arctic sea ice: set `sea_ice_cover` to 0 (and decide what SST the newly open water should have!).
* Put a warm anomaly at the equator and compare it with Part 3.

Fields in the initial state: `IC` (an xarray Dataset). Use `add_to(ds, variable, anomaly)` to make a modified copy and `run_forecast(ic, days)` to run it.

In [ ]:
#@title ✏️ Example: weaken the Northern Hemisphere jet by 30% between 30–60°N
my_days = 10
band = ((IC.latitude >= 30) & (IC.latitude <= 60)).astype(float)
u = IC["u_component_of_wind"]
IC_mine = add_to(IC, "u_component_of_wind", -0.3 * u * band)

mine = run_forecast(IC_mine, my_days)
ctrl = run_forecast(IC, my_days)

show = [0, 2, 5, my_days]
plot_maps([z500(mine).sel(lead_day=d) - z500(ctrl).sel(lead_day=d) for d in show],
          [f"Z500 difference, day {d}" for d in show],
          contours=[z500(ctrl).sel(lead_day=d) for d in show],
          cbar_label="Experiment - control Z500 (m)")

**Write-up:** state your hypothesis, show one or two figures, and explain whether the model behaved as you expected. Also say whether the response is physically sensible, or a sign that the model is being pushed outside the states it was trained on.

### References
* Kochkov et al. (2024), Neural general circulation models for weather and climate, *Nature* 632, 1060–1066.
* Selz & Craig (2023), Can artificial intelligence-based weather prediction models simulate the butterfly effect?, *GRL* 50, e2023GL105747.
* Hakim & Masanam (2024), Dynamical tests of a deep learning weather prediction model, *AIES* 3, e230090.
* NeuralGCM documentation: https://neuralgcm.readthedocs.io